In [161]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException
from selenium.webdriver.chrome.options import Options
import time
import os
import pandas as pd

In [162]:
def set_up_driver():
    chrome_options = Options()
    #chrome_options.add_argument("--headless") 
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("start-maximized")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled") # Tắt tính năng phát hiện tự động của trình duyệt
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"]) # Loại bỏ thông báo "Chrome is being controlled by automated test software"
    chrome_options.add_experimental_option('useAutomationExtension', False) # Vô hiệu hóa tiện ích mở rộng tự động hóa
    driver = webdriver.Chrome(options=chrome_options)
    return driver

In [163]:
def get_medicine_category(driver):
    category = []
    try:

        category_section = driver.find_element(By.CSS_SELECTOR, "#collapseFilter-0 > div")
        time.sleep(1)

        category_elements = category_section.find_elements(By.CSS_SELECTOR, "#collapseFilter-0 ul li")

        if not category_elements:
            print("Không tìm thấy danh mục nào.")
            return []

        for element in category_elements:
            # Lấy label trong chính li
            label = element.find_element(By.CSS_SELECTOR, "label.form-check-label")
            text = label.text.strip()

            if text:
                checkbox = element.find_element(By.CSS_SELECTOR, "input.form-check-input[type='checkbox']")
                value = checkbox.get_attribute("value")

                category.append({
                    "category": text,
                    "value": value
                })

    except Exception as e:
        print(f"Lỗi khi lấy danh mục: {e}")
        return []

    return category

In [164]:

from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains

def focus_main_page(driver):
    driver.execute_script("""
        if (document.activeElement) document.activeElement.blur();
        document.body.setAttribute('tabindex', '-1');
        document.body.focus();
    """)
    time.sleep(0.1)


def scroll_down_by_key(driver, times=5, delay=0.3):
    body = driver.find_element(By.TAG_NAME, "body")
    for _ in range(times):
        body.send_keys(Keys.PAGE_DOWN)
        time.sleep(delay)

In [ ]:

def get_medicine_link(driver, category_value, category_name):
    links = []
    wait = WebDriverWait(driver, 10)

    try:
        checkbox = wait.until(EC.element_to_be_clickable(
            (By.CSS_SELECTOR, f"input.form-check-input[value='{category_value}']")
        ))
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", checkbox)
        driver.execute_script("arguments[0].click();", checkbox)
        print(f"Đã click vào category: {category_name}")
        focus_main_page(driver)

        # Chờ list sản phẩm xuất hiện
        wait.until(EC.presence_of_all_elements_located(
            (By.CSS_SELECTOR, "#loadDataAjaxPage .itemsanpham")
        ))
        time.sleep(1)

        # Click "Xem thêm" đến khi hết
        max_click = 80
        for _ in range(max_click):
            scroll_down_by_key(driver, times=6, delay=0.25)
            count_before = len(driver.find_elements(By.CSS_SELECTOR, "#loadDataAjaxPage .itemsanpham"))

            try:
                load_more_btn = WebDriverWait(driver, 3).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, "#idloadpagemore > button.p-3.border-0.px-3.py-2"))
                )
            except TimeoutException:
                print("Không còn nút Xem thêm.")
                break

            if not load_more_btn.is_displayed() or not load_more_btn.is_enabled():
                print("Nút Xem thêm không còn khả dụng.")
                break

            driver.execute_script("arguments[0].click();", load_more_btn)

            # Chờ số item tăng sau khi click
            try:
                WebDriverWait(driver, 2).until(
                    lambda d: len(d.find_elements(By.CSS_SELECTOR, "#loadDataAjaxPage .itemsanpham")) > count_before
                )
            except TimeoutException:
                # Không tăng nữa => đã hết dữ liệu
                print("Đã load hết sản phẩm cho category này.")
                break

        # Lấy link sản phẩm
        product_anchors = driver.find_elements(By.CSS_SELECTOR, "#loadDataAjaxPage .itemsanpham a[href]")
        seen = set()
        for a in product_anchors:
            href = a.get_attribute("href")
            if href and href not in seen:
                seen.add(href)
                links.append({
                    "category": category_name,
                    "link": href
                })

        print(f"Tổng: {len(links)} sản phẩm trong category [{category_name}]")

        # Bỏ chọn category
        checkbox = driver.find_element(By.CSS_SELECTOR, f"input.form-check-input[value='{category_value}']")
        driver.execute_script("arguments[0].click();", checkbox)
        time.sleep(0.8)

    except Exception as e:
        print(f"Lỗi khi lấy link category [{category_name}]: {type(e).__name__}: {e}")

    return links

In [166]:

def crawl_minhchau_links():
    driver = set_up_driver()
    wait = WebDriverWait(driver, 3)
    records = []

    try:
        print("Đang truy cập trang web...")
        driver.get("https://nhathuocminhchau.com/")
        time.sleep(5)

        print("Đang tìm kiếm danh mục thuốc...")
    
        original_tab = driver.current_window_handle

        thuoc_page = wait.until(EC.element_to_be_clickable(
            (By.XPATH, "//a[contains(@href, '/thuoc')]"))
        )
        thuoc_page.click()
        time.sleep(2)

        # Kiểm tra nếu mở tab mới thì switch sang tab mới
        all_tabs = driver.window_handles
        if len(all_tabs) > 1:
            new_tab = [tab for tab in all_tabs if tab != original_tab][0]
            driver.switch_to.window(new_tab)
            print(f"Đã switch sang tab mới: {driver.current_url}")
        else:
            # Không mở tab mới thì điều hướng thẳng
            driver.get("https://nhathuocminhchau.com.vn/thuoc/")
            print(f"Điều hướng tới: {driver.current_url}")

        time.sleep(3)

        menu = get_medicine_category(driver)
        print(f"Đã tìm thấy {len(menu)} danh mục thuốc.")
        for element in menu:
            print(f"Category: {element['category']} - Value: {element['value']}")


        for i, item in enumerate(menu):
            print(f"\n[{i+1}/{len(menu)}] Đang xử lý: {item['category']}")
            links = get_medicine_link(driver, item['value'], item['category'])
            records.extend(links)
            print(f"Tổng link đã lấy: {len(records)}")

            #Lưu tạm sau mỗi category đề phòng lỗi
            df_temp = pd.DataFrame(records)
            df_temp.to_csv("medicine_links_temp.csv", index=False, encoding="utf-8-sig")

        # Lưu file CSV cuối cùng
        df = pd.DataFrame(records)
        df.to_csv("medicine_links.csv", index=False, encoding="utf-8-sig")
        print(f"\nĐã lưu {len(records)} link vào medicine_links.csv")

    except Exception as e:
        print(f"Đã xảy ra lỗi: {type(e).__name__}: {e}")
    finally:
        driver.quit()


In [ ]:

test = crawl_minhchau_links()

Đang truy cập trang web...
Đang tìm kiếm danh mục thuốc...
Đã switch sang tab mới: https://nhathuocminhchau.com.vn/thuoc/
Đã tìm thấy 33 danh mục thuốc.
Category: Thuốc kháng sinh, Kháng nấm - Value: 000310032001
Category: Thuốc tim mạch & Huyết áp - Value: 000310032003
Category: Thuốc tiểu đường - Value: 000310032004
Category: Thuốc hướng thần & Cai nghiện - Value: 000310032005
Category: Thuốc chống dị ứng ( kháng histamin) - Value: 000310032006
Category: Thuốc dùng ngoài - Value: 000310032007
Category: Thuốc Hô Hấp - Value: 000310032008
Category: Thuốc kháng viêm, giảm đau & hạ sốt - Value: 000310032010
Category: Thuốc cường dương - Value: 000310032011
Category: Thuốc Tiêu Hóa, gan mật - Value: 000310032012
Category: Thuốc trị ung thư, u bướu - Value: 000310032014
Category: Thuốc giãn mạch - Value: 000310032015
Category: Thuốc Hocmon, Nội tiết tố - Value: 000310032018
Category: Thuốc tiêm, dịch truyền - Value: 000310032020
Category: Thuốc đường tiết niệu - Value: 000310032029
Categor